# Session 28: GRU - Part 2 (Hands-on & Comparison)


## Task 1: Prepare Sequence Dataset (Stock Prices)
We will use `yfinance` to download historical stock data (Apple - AAPL)
*(Note:`!pip install yfinance` in your environment first).*

In [ ]:
!pip install yfinance

In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

# 1. Download Stock Data (AAPL)
print("Downloading data...")
df = yf.download('AAPL', start='2020-01-01', end='2023-01-01')
data = df['Close'].values.reshape(-1, 1)

# 2. Normalize the data (important for Neural Networks)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# 3. Create Sequences (Sliding Window)
window_size = 10
X = []
y = []

for i in range(window_size, len(scaled_data)):
    X.append(scaled_data[i-window_size:i, 0])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)

# 4. Reshape X for Keras (Samples, Time Steps, Features)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

print(f"Data Prepared! X shape: {X.shape}, y shape: {y.shape}")

[*********************100%***********************]  1 of 1 completed

Data Prepared! X shape: (746, 10, 1), y shape: (746,)


## Task 2 & 3: Build, Train, and Benchmark GRU vs. LSTM
We will define both models with identical architectures (apart from the recurrent layer type) to ensure a fair comparison. We use Mean Squared Error (MSE) because stock price prediction is a regression task, not classification.

In [6]:
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense

# --- Build GRU Model ---
gru_model = Sequential([
    GRU(50, input_shape=(X.shape[1], 1)),
    Dense(1)
])
gru_model.compile(optimizer='adam', loss='mse')

# --- Build LSTM Model ---
lstm_model = Sequential([
    LSTM(50, input_shape=(X.shape[1], 1)),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

# --- Train and Time GRU ---
print("--- Training GRU ---")
start_time = time.time()
# verbose=1 prints the loss after each epoch as requested
gru_history = gru_model.fit(X, y, epochs=10, batch_size=32, verbose=1) 
gru_time = time.time() - start_time
gru_loss = gru_history.history['loss'][-1]
gru_params = gru_model.count_params()

# --- Train and Time LSTM ---
print("\n--- Training LSTM ---")
start_time = time.time()
lstm_history = lstm_model.fit(X, y, epochs=10, batch_size=32, verbose=1)
lstm_time = time.time() - start_time
lstm_loss = lstm_history.history['loss'][-1]
lstm_params = lstm_model.count_params()

print("\nTraining Complete!")

c:\Users\param\machine learning\deep learning\CNN project\cnn\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--- Training GRU ---
Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.1528
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0149
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0074
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0048
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0029
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0018
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0013
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0010    
Epoch 9/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.9902e-04
Epoch 10/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.6336e-04

--- Training LSTM ---
Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1028
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0062
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0030
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0021
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━

## Task 4: Comparison Table & Analysis


In [7]:
import pandas as pd

results = {
    "Model Type": ["GRU", "LSTM"],
    "Total Parameters": [gru_params, lstm_params],
    "Training Time (sec)": [round(gru_time, 2), round(lstm_time, 2)],
    "Final Loss (MSE)": [round(gru_loss, 5), round(lstm_loss, 5)]
}

df_results = pd.DataFrame(results)
display(df_results)

print("\n--- Brief Explanation ---")
print("The GRU model typically trains faster and has fewer parameters (due to having 2 gates instead of 3).")
print("For relatively simple sequences like this 10-day sliding window, both models often achieve very similar loss.")
print("Therefore, the GRU is usually the better choice here due to its computational efficiency.")

,Model Type,Total Parameters,Training Time (sec),Final Loss (MSE)
0,GRU,8001,4.20,0.00096
1,LSTM,10451,3.52,0.00147



--- Brief Explanation ---
The GRU model typically trains faster and has fewer parameters (due to having 2 gates instead of 3).
For relatively simple sequences like this 10-day sliding window, both models often achieve very similar loss.
Therefore, the GRU is usually the better choice here due to its computational efficiency.


## Task 5: When to prefer GRU over LSTM (Real-world Scenarios)

Because GRUs have fewer parameters and require less memory and computational power than LSTMs, they are highly preferred in **resource-constrained environments** or when **training time is a critical bottleneck**, assuming the sequence dependencies aren't aggressively long. 

Here are 3 real-world examples (excluding stock prices):

1. **Wake-Word Detection on Smart Speakers (e.g., "Hey Siri" / "Alexa"):**
   * **Reasoning:** Smart speakers need to run sequence models locally on low-power edge devices. A GRU uses less battery and memory while still perfectly handling the short audio sequence needed to recognize a wake word.
2. **Predictive Text / Autocomplete on Mobile Keyboards:**
   * **Reasoning:** Mobile phones have strict RAM limitations. A GRU can quickly process the sequence of the last few typed words and suggest the next word with minimal latency, providing a snappy user experience without draining the phone's battery like a heavier LSTM might.
3. **Real-time IoT Sensor Anomaly Detection (e.g., Factory machinery vibrations):**
   * **Reasoning:** Factory sensors generate constant streams of time-series data. Running a lightweight GRU on a microcontroller attached directly to the machine allows for instant processing and anomaly alerts without the heavy compute requirements of an LSTM or sending data to the cloud.